# Pilot solver runs on Colab

This notebook is a **frozen launcher** — it should never need editing.
The actual experiment definition (which arms, which flags) lives in
`metaculus/run_pilot.py` in the repo. To change the experiment: edit that
file locally, `git commit -am wip && git push`, then rerun the resync cell
and the payload cell here.

Solver fits reuse the committed LLM cache (`metaculus/caches/`)
— no API keys needed. Each arm is resume-safe. To parallelize across k
sessions, pass `--shard i/k` to the payload cell (outputs get a per-shard
suffix; merge with the last cell).

In [ ]:
!git clone https://github.com/amdson/calibrated_response.git
%cd calibrated_response
%pip install -q -e .

In [ ]:
# RESYNC: after a local `git commit -am wip && git push`, rerun this cell —
# no re-clone or reinstall needed (editable install picks up code changes).
# Untracked outputs (arm_*.json) survive the reset.
!git fetch origin && git reset --hard origin/main

In [ ]:
# use the GPU when the runtime has one (the solver script defaults to CPU
# only when JAX_PLATFORMS is unset)
import os, subprocess
has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
os.environ["JAX_PLATFORMS"] = "cuda" if has_gpu else "cpu"
print("JAX_PLATFORMS =", os.environ["JAX_PLATFORMS"])

In [ ]:
# PAYLOAD: runs whatever metaculus/run_pilot.py currently defines
# (all arms + diagnostics). Extra args are forwarded to every arm,
# e.g.  --shard 0/2   or   --limit 5  for a smoke test.
!python metaculus/run_pilot.py

In [ ]:
# OPTIONAL: merge per-shard outputs into the unsuffixed arm files
# import json, glob, re
# for p in glob.glob("metaculus/runs/*/pred_*_shard*.json"):
#     tgt = re.sub(r"_shard\d+", "", p)
#     merged = json.load(open(tgt)) if glob.glob(tgt) else {}
#     merged.update(json.load(open(p)))
#     json.dump(merged, open(tgt, "w"), indent=1)
#     print(tgt, len(merged), "rows")


In [ ]:
# get the results off the runtime — predictions live in the run dir
# (metaculus/runs/ is gitignored apart from manifests, so nothing here
# can be resurrected by the resync cell's git reset)
import glob
from google.colab import files
for p in sorted(glob.glob("metaculus/runs/*/pred_*.json")
                + glob.glob("metaculus/runs/*/manifest.json")):
    files.download(p)
# ...or mount Drive and copy:
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r metaculus/runs /content/drive/MyDrive/
